In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from domain_funcs import nam_domain_outline, core_site_boxes

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
#np.set_printoptions(threshold=np.inf) # disable truncation
import pandas as pd 
from scipy.stats import ttest_rel, ttest_ind  # used by sigtest()/sigtest2n() below

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, ListedColormap, LinearSegmentedColormap
from matplotlib import cm
import cmocean.cm as cmo
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory (override with the WORK_DATA_DIR env var; see config/paths.env.example)
dpath0=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')
# save figs here, under a subdir named for this notebook. HPC has no path to OneDrive, so this
# is a separate directory from the laptop-side FIG_OUTPUT_DIR (config/paths.env.example,
# proxy-side only) -- sync the two by hand (rsync/scp) when picking work back up on the laptop.
# Override the root with HPC_FIG_OUTPUT_DIR; defaults to a figures/ subdir next to the PI/,
# LGM/, obs_data/ dirs under WORK_DATA_DIR.
opath=os.path.join(os.environ.get('HPC_FIG_OUTPUT_DIR', f'{dpath0}/notebooks/outputs'))
os.makedirs(opath, exist_ok=True)

In [ ]:
# student's t-test (for data with identical sample sizes)
def sigtest(yearmean1,yearmean2,timemean1,timemean2):
    ptvals = ttest_rel(yearmean1,yearmean2, axis=0)
    diff = timemean1-timemean2
    diff_mask = np.ma.masked_where(ptvals[1] > 0.1,diff)
    return diff, diff_mask, ptvals

# Welch's t-test (for data with different sample sizes)
def sigtest2n(yearmean1,yearmean2,timemean1,timemean2):
    ptvals = ttest_ind(yearmean1,yearmean2, axis=0, equal_var = False)
    diff = timemean1-timemean2
    diff_mask = np.ma.masked_where(ptvals[1] > 0.1,diff)
    return diff, diff_mask, ptvals

def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd

In [ ]:
#=== SET FILE PATH INFO

files = {}

files['pi'] = {}
for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
    files['pi'][varn] = f'{dpath0}/PI/dh.precIsotopes.atm.iPI.nc'
for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
    files['pi'][varn] = f'{dpath0}/PI/o.precIsotopes.atm.iPI.nc'
for varn in ['PRECC', 'PRECL', 'TS', 'PSL']:
    files['pi'][varn] = f'{dpath0}/PI/b.e12.B1850C5.f19_g16.iPI.01.400-499.climo.nc'
for varn in ['Q','U','V','OMEGA', 'Z3']: # these are in their own files because I converted the pressure levels from hybrid to standard
    files['pi'][varn] = f'{dpath0}/PI/{varn}.iPI.0400-0499.climo.nc'

files['lgm'] = {}
for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS',
            'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS',
            'PRECC', 'PRECL', 'TS', 'PSL']:
    files['lgm'][varn] = f'{dpath0}/LGM/b.e12.B1850C5.f19_g16.i21ka.03.cam.h0.0801-0900.climo.nc'
for varn in ['Q','U','V','OMEGA', 'Z3']: # these are in their own files because I converted the pressure levels from hybrid to standard
    files['lgm'][varn] = f'{dpath0}/LGM/{varn}.i21ka.0801-0900.climo.nc'


In [ ]:
# --- PROCESS iCESM1.2 OUTPUT --- #

cases=['pi','lgm']

#== load raw variables
raw_varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS', 
             'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS', 
             'PRECC', 'PRECL','TS','U','V','OMEGA','Q','Z3','PSL']
raw = { case: { varn:{} for varn in raw_varns } for case in cases}

for case in cases:
    for varn in raw_varns:
        raw[case][varn]=xr.open_dataset(files[case][varn])[varn]
        raw[case][varn].attrs['original_time_values'] = raw[case][varn].time
        # update time axis?
        raw[case][varn]=raw[case][varn].rename({'time':'month'})
        raw[case][varn]=raw[case][varn].assign_coords(month=[1,2,3,4,5,6,7,8,9,10,11,12])


#== construct processed data dictionary
dat_varns = ['TS', 'OMEGA', 'U', 'V', 'PSL', 'Q', 'Z3', 'PRECC', 'PRECL', 'PRECT', 'dDp', 'd18Op']
dat = { case: { varn:{} for varn in raw_varns } for case in cases}

for case in cases:
    for varn in dat_varns:
        if varn in ['TS', 'OMEGA', 'U', 'V', 'PSL', 'Q', 'Z3']:
            dat[case][varn] = raw[case][varn]
        elif varn in ['PRECC', 'PRECL']:
            dat[case][varn] = raw[case][varn]*1000*60*60*24 # convert from m/s to mm/day
            dat[case][varn].attrs['units'] = 'mm/day'
        elif varn in ['PRECT']:
            # calculate total precip from convective and large-scale prec vars (snow+rain).
            dat[case][varn] = dat[case]['PRECC'] + dat[case]['PRECL']
            dat[case][varn].attrs['units'] = 'mm/day'
            dat[case][varn].attrs['long_name'] = 'total precipitation'
            dat[case][varn].attrs['source'] = 'PRECC + PRECL'
        else:
            #== calculate precipitation-weighted fields
            # calculate precipitation weights by month
            annual_total_p = dat[case]['PRECT'].sum(dim="month")
            pWeights = dat[case]['PRECT']/annual_total_p
            # the ptiny constant floors the denominator of the isotope ratio equation to prevent divide by zero
            ptiny=1e-18
            if varn=='dDp':
                # Hydrogen
                phyd = raw[case]['PRECRC_H2Or'] + raw[case]['PRECRL_H2OR'] + raw[case]['PRECSC_H2Os'] + raw[case]['PRECSL_H2OS']
                pdeu = raw[case]['PRECRC_HDOr'] + raw[case]['PRECRL_HDOR'] + raw[case]['PRECSC_HDOs'] + raw[case]['PRECSL_HDOS']
                # replace very small ph values with a tiny value
                phyd = phyd.where(phyd > ptiny, ptiny) 
                # turn into per mil notation
                dd = (pdeu/phyd - 1)*1000 
                # Multiply isotope values by weights
                dat[case][varn] = dd*pWeights
            elif varn=='d18Op':
                # Oxygen
                p16o = raw[case]['PRECRC_H216Or'] + raw[case]['PRECRL_H216OR'] + raw[case]['PRECSC_H216Os'] + raw[case]['PRECSL_H216OS']
                p18o = raw[case]['PRECRC_H218Or'] + raw[case]['PRECRL_H218OR'] + raw[case]['PRECSC_H218Os'] + raw[case]['PRECSL_H218OS']
                # replace very small ph values with a tiny value
                p16o = p16o.where(p16o > ptiny, ptiny)
                # turn into per mil notation
                do = (p18o/p16o - 1)*1000 
                # Multiply isotope values by weights
                dat[case][varn] = do*pWeights
            else:
                print(f'{varn} not recognized')


#== Calculate LGM-PI differences across variables
var_diffs = {varn: dat['lgm'][varn] - dat['pi'][varn] for varn in dat_varns}


#== iCESM1.2 topography
in_file = '/glade/u/home/dervlamk/nam-interglacial-dD/data/raw/topo_21ka_remap_19x25.mod.170428.sm9.nc'
LANDFR = xr.open_dataset(in_file).LANDFRAC
PHIS = xr.open_dataset(in_file).PHIS
# the available variable is "surface geopotential" in units m2/s2
# approximate the surface geometric elevation by dividing by gravitational acceleration
g = 9.80665 # m/s2
zsurf = PHIS / g
zsurf.attrs['units'] = 'm'
zsurf.attrs['long_name'] = 'surface elevation'
zsurf.attrs['source_file'] = '/glade/work/jiangzhu/data/inputdata/cesm120ka_ICEG6/21ka/topo_21ka_remap_19x25.mod.170428.sm9.nc'

In [ ]:
# --- LOAD OTHER RELEVANT DATA --- #

# IMERG precipitation
climo_filen = f'{dpath0}/obs_data/imerg.gn.2001-2018.climo.nc'
if os.path.exists(climo_filen):
    ds = xr.open_dataset(climo_filen).precipitation
else:
    filen = f'{dpath0}/obs_data/imerg.gn.timeseries.2001-2018.nc'
    ds = xr.open_dataset(filen).precipitation.transpose('time','lat','lon').groupby("time.month").mean(dim='time') * 24 # convert from mm/hr to mm/day
    ds.attrs['units'] = 'mm/day'
    ds.attrs['Units'] = 'mm/day'
    # Convert to 0:360
    nx   = len(ds.lon)
    lons = np.linspace(0,360,nx)
    ds['lon'] = lons
    ds=ds.roll(lon=3600)
    ds.attrs['source_filename'] = 'imerg.gn.timeseries.2001-2018.nc'
    ds.to_netcdf(climo_filen, mode='w')

# ETOPO05 topography
filen = f'{dpath0}/obs_data/obs.etopo5.zsurf.nc'
etopo_full = xr.open_dataset(f'{filen}').ROSE
etopoSWNA = etopo_full.sel(ETOPO05_X=slice(235,275), ETOPO05_Y=slice(10,42))

# Proxy timeslice mean values
proxydD = pd.read_csv('../data/processed/timeslice_mean_proxy_dDp.csv')

## Statistical significance of LGM$-$PI differences -- blocked, not abandoned

The two cells below (commented out) sketch how to test whether the LGM$-$PI differences
plotted in this notebook are statistically significant, rather than just reporting the raw
climatological difference. **They don't run today and shouldn't be un-commented as-is.**

Why: `sigtest()`/`sigtest2n()` (see the function defs near the top of this notebook) need a
per-year sample -- e.g. `ann_seas_mean[sim]` with a `year` dimension -- to run a paired/unpaired
t-test against. The `pi` and `lgm` files this notebook actually loads
(`b.e12.B1850C5.f19_g16.i21ka.03.cam.h0.0801-0900.climo.nc` and
`b.e12.B1850C5.f19_g16.iPI.01.400-499.climo.nc`) are **pre-computed monthly climatologies**
shared by a collaborator -- there is no per-year timeseries behind them on this end, so there is
nothing to compute interannual variance from, and no significance test can be built from what's
currently on disk.

**Action item:** ask the collaborator for the original per-year (or per-month, uncollapsed)
output underlying these two climo files, not just the climo files themselves. Once that exists:
- Cell 7 (`+++ CALCULATE TIME-MEANS +++`) is a reasonable starting point for building
  `ann_seas_mean['pi']` / `ann_seas_mean['lgm']`, but references undefined `key`/`run`/`mons`
  and a truncated variable name (`for var in ['d`) -- it needs rewriting against this
  notebook's actual `dat['pi']`/`dat['lgm']` structure, not adapting in place.
- Cell 8 (`+++ COMPARING ALL MODEL RUNS TO MODEL CTRL. +++`) is copy-pasted from a different
  notebook entirely -- `flor`, `flor_mod_runs`, `hicam`/`hitopo` are runs from another project's
  model comparison, not this one's `pi`/`lgm` pair. Treat it only as a worked example of the
  `sigtest()` call signature, not as code to adapt piecemeal.

In [ ]:
"""
### +++ CALCULATE TIME-MEANS +++ ###

seasons=['DJF','JFM','JAS','JJAS']
seas_mean = { 'pi': {}, 'lgm': {} }
ann_seas_mean = { 'pi': {}, 'lgm': {} }

print('Calculating seasonal means for...')
for sim in ['pi','lgm']:
    print(sim)
    for var in ['d
    for season in seasons:
        months = get_season(season=season)
        # seasonal mean for whole timeseries
        custom_seasons = xr.where(dat[sim][var]['time'].dt.month.isin(mons), season, 'Other')
        seas_mean[key][run] = dat[key][run].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
        # seasonal mean by year
        ann_seas_mean[key][run] = dat[key][run].sel(time=dat[key][run]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')

print('Done.')
"""

In [ ]:
"""
### +++ COMPARING ALL MODEL RUNS TO MODEL CTRL. +++ ###

# initialize dictionaries
model_diff      = { 'flor' : {} }
model_diff_mask = { 'flor' : {} }
model_ptvals    = { 'flor' : {} }

# names of modified topography runs
flor_mod_runs=['hicam','hitopo']

print('Significance testing for:')
for key in model_diff.keys():
    print(f'{key}')
    for run in flor_mod_runs:
        print(f'...{run}')
        # calculate significance of model - obs difference
        diff_, diff_mask_, ptvals_ = sigtest(ann_jas_mean[key][run], ann_jas_mean[key]['ctrl'],
                                             jas_mean[key][run], jas_mean[key]['ctrl'])
        model_diff[key][run] = diff_
        model_diff_mask[key][run] = diff_mask_
        model_ptvals[key][run] = ptvals_

print('Done.')
"""

## Core-site box means (LGM$-$PI)

In [ ]:
# --- CORE-SITE MEAN LGM-PI dD_precip --- #

# The boxes are defined once, in scripts/py_functions/domain_funcs.py -> core_site_boxes(), and
# are the same object drawn on the maps below (ax.add_geometries(...)) -- so the region shown
# and the region averaged here cannot drift apart. Same box definitions and same cos(lat)
# weighted-mean approach as swna_modern_climatology.ipynb, which averages the OIPC isoscape over
# these boxes. Unlike OIPC (a terrestrial-only isoscape), the model grid has no ocean mask to
# worry about, so this is a plain weighted mean over the box, no NaN-skipping needed.
#
# The model grid is 0:360 in longitude; core_site_boxes() is -180:180 (matching the proxy lon/
# lat columns and how boxes are drawn on these PlateCarree maps), so bounds are converted with
# `% 360` before slicing.

site_boxes = core_site_boxes()
seasons = ['ann', 'jas']

site_dDdiff = {site: {} for site in site_boxes}
for site, poly in site_boxes.items():
    w, s, e, n = poly.bounds  # shapely: (min_lon, min_lat, max_lon, max_lat)
    box = var_diffs['dDp'].sel(lon=slice(w % 360, e % 360), lat=slice(s, n))
    for season in seasons:
        seas_box = box.isel(month=get_season(season)).mean(dim='month')
        weights = np.cos(np.deg2rad(seas_box.lat))
        site_dDdiff[site][season] = float(seas_box.weighted(weights).mean(('lat', 'lon')))

print('Model LGM-PI delta-dD_precip [per mil] -- cos(lat)-weighted mean over each core-site box\n')
print(f"{'season':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
for season in seasons:
    print(f'{season:>8}' + ''.join(f"{site_dDdiff[site][season]:>12.2f}" for site in site_boxes))

# FIGS

### JAS only

In [ ]:
# Proxy Data (based on C30 handpicked data as of Aug 2025)
clons=proxydD['lon'].values
clats=proxydD['lat'].values
ddiff = proxydD['lgm_dD'].values - proxydD['late_holocene_dD'].values
# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
im=6 # June
em=9 # Sept
# NAM domain outline, shared with swna_modern_climatology.ipynb (scripts/py_functions/
# domain_funcs.py). site_boxes comes from the core-site box means cell above -- same object,
# so what's drawn here and what's averaged there can't drift apart.
nam_domain = nam_domain_outline()
# plot specs
bbox={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':18, 'ha':'center', 'va':'bottom'}
titles = np.array([
    r'$\mathbf{\Delta}$Precipitation',                # ΔPrecipitation
    r'$\mathbf{\Delta \delta} \mathbf{D_{precip}}$',  # ΔδD_precip
    r'$\mathbf{\Delta \omega_{500}}$ and 850 mb Wind'  # Δω_500 & 850mb WIND
])
months = {0: 'January-February-March', 5: 'June-July-August-September', 6: 'July-August-September'}
t_months = months.get(im, '<not_defined>')
letters = np.array(['a','b','c'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -82.5, 10., 36.]
# vector specs
skip_n=1
w=0.0075
scalef=10
key_length=2

# per-panel data + color specs -- looped over below instead of one block of code per panel
panels = [
    dict(data=var_diffs['PRECT'][im:em].mean(dim="month"), cmap=get_settings(field='precip', diff=True)[0],
         vmin=-2.5, vmax=2.5, nlevels=21, cbar_ticks=[-2,-1,0,1,2],
         cbar_label='[mm day$^{-1}$]', proxy_colored=False, quiver=False),
    dict(data=var_diffs['dDp'][im:em].mean(dim="month"), cmap=cm.RdBu_r, vmin=-3, vmax=3, nlevels=25,
         cbar_ticks=[-3,-2,-1,0,1,2,3], cbar_label=u'[‰]', proxy_colored=True, quiver=False),
    dict(data=var_diffs['OMEGA'].sel(lev_p=500.0)[im:em].mean(dim="month"), cmap=cmo.balance,
         vmin=-0.05, vmax=0.05, nlevels=21, cbar_ticks=[-0.04,-0.02,0,0.02,0.04],
         cbar_label='[Pa s$^{-1}$]', proxy_colored=False, quiver=True),
]
for p in panels:
    p['norm'] = mpl.colors.BoundaryNorm(np.linspace(p['vmin'], p['vmax'], p['nlevels']), p['cmap'].N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16,4.5), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1,t_months+' iCESM1.2 LGM (21ka) $-$ PI', **text_kw)

for i, (axi, p) in enumerate(zip(ax, panels)):
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, titles[i], **text_kw)
    p['cf'] = axi.pcolormesh(lon, lat, p['data'], cmap=p['cmap'], norm=p['norm'], transform=trans)

    if p['quiver']:
        u850 = var_diffs['U'].sel(lev_p=850.0)[im:em].mean(dim="month")
        v850 = var_diffs['V'].sel(lev_p=850.0)[im:em].mean(dim="month")
        q1 = axi.quiver(lon[::skip_n], lat[::skip_n], u850[::skip_n,::skip_n], v850[::skip_n,::skip_n],
                         color='k', width=w, scale=scalef, scale_units='inches', units='height',
                         transform=trans, zorder=100)
        axi.quiverkey(q1, .95, 1.035, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='W',
                       fontproperties={'size':9})

    if p['proxy_colored']:
        axi.scatter(x=clons, y=clats, c=ddiff, cmap=p['cmap'], norm=p['norm'], alpha=1, edgecolor='k',
                     s=150, transform=trans, zorder=100)
        for j in [0,1]:
            axi.text(clons[j]-1.1, clats[j], f'{ddiff[j]:.1f}‰', fontsize=10, weight='bold', ha='right',
                      bbox=bbox, zorder=100)
    else:
        axi.scatter(clons, clats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    # map dressing, common to all three panels
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    #axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=[1], linewidths=1.2, colors='k', transform=trans)
    axi.coastlines(lw=1)
    #axi.add_feature(cfeature.BORDERS, lw=1)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)
    # NAM domain outline, all panels (red, replacing the old ad hoc black box)
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # core-site sampling boxes, dDp panel only -- the other two panels don't need them
    if i == 1:
        axi.add_geometries(list(site_boxes.values()), crs=trans, fc='none', ec='k', lw=1,
                            linestyle='--', zorder=9)
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

for i, p in enumerate(panels):
    cax = fig.add_axes([0.05 + i*0.325, 0, 0.275, 0.05])
    cbar = fig.colorbar(p['cf'], ticks=p['cbar_ticks'], orientation='horizontal', extend='both', cax=cax)
    cbar.set_label(p['cbar_label'], weight='normal', labelpad=5, rotation=0)
    cbar.ax.tick_params(labelsize=10)
    for tick in cbar.ax.xaxis.get_major_ticks():
        tick.label1.set_fontweight('normal')

plt.savefig(os.path.join(opath, "LGM-PI_icesm1p2_diffs.png"), dpi=1200, bbox_inches='tight')

## Convective vs. Large-Scale Precip

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':24, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'center', 'rotation':90}
titles=np.array([r'PRECC', r'PRECL'])
seasons=np.array(['ANN','JFM', 'JAS'])
letters=['A','B','C','D','E','F']
tx=-101.75
ty=36
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -82.5, 10., 36.]
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-2.5
pvmax=2.5
plevels=np.linspace(pvmin, pvmax, 21)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(20,10), layout='constrained', subplot_kw={'projection':proj})

for i in [0,1,2]:
    ax[0,i].text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+1, seasons[i], **text_kw)
    if 0 <= i <= 1:
        ax[i,0].text(map_bnds[0]-7, map_bnds[2]+((map_bnds[3]-map_bnds[2])/2), titles[i], **text_kw2)
    else:
        pass

ax[0,0].pcolormesh(pcdiff.lon, pcdiff.lat, pcdiff.mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[0,1].pcolormesh(pcdiff.lon, pcdiff.lat, pcdiff[0:3,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[0,2].pcolormesh(pcdiff.lon, pcdiff.lat, pcdiff[6:9,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)

ax[1,0].pcolormesh(pcdiff.lon, pcdiff.lat, pldiff.mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[1,1].pcolormesh(pcdiff.lon, pcdiff.lat, pldiff[0:3,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
cf=ax[1,2].pcolormesh(pcdiff.lon, pcdiff.lat, pldiff[6:9,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)

for i, ax in enumerate(ax.flat): 
    # add box around core NAM domain
    lon_bnds = np.array([-113, -104, -104, -113])
    lat_bnds = np.array([20,  20,  34,  34])
    ring=LinearRing(list(zip(lon_bnds, lat_bnds)))
    #ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=3, linestyle='--', zorder=11) 
    # subplot labels
    ax.text(-121, ty+1, letters[i], **text_kw1)
    # map properties
    ax.coastlines(color='k', linewidth=2)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=2)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=2)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=[-3,-2,-1,0,1,2,3], orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$ PRECIPITATION [mm/day]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

plt.savefig(os.path.join(opath, "cesm1.2_LGM-PI_precc_precl_diffs.pdf"), bbox_inches='tight')

## Moisture Transport

In [ ]:
#=== Calculate moisture transport

qv = { 'pi':{}, 'lgm':{} }
qu = { 'pi':{}, 'lgm':{} }
mt = { 'pi':{}, 'lgm':{} }

for key in ['pi','lgm']:
    qu[key] = dat[key]['U']*dat[key]['Q']
    qv[key] = dat[key]['V']*dat[key]['Q']
    mt[key] = windSpd(qu[key], qv[key]) 